In [ ]:
from utils.consts import (
    SHARED_DIR_PATH,
    SPARK_CONFIGS
)

In [ ]:
from run_utils import (
    prepare_data, build_cvae_lstm_model, train_model,
    save_artifacts, load_artifacts,
    generate_timespan, generate_n_weeks,
)

In [ ]:
TRAINING_DATA_PATH = SHARED_DIR_PATH / "training_data" / "greg_tmp"
ARTIFACTS_DIR_PATH = SHARED_DIR_PATH / "artifacts"
# RUN_DIR_PATH  = ARTIFACTS_DIR_PATH / "run_1"
WEIGHTS_PATH  = RUN_DIR_PATH / "models_weights"

In [ ]:
# MODEL_PATH  = WEIGHTS_PATH / "cvae_lstm_v2_0.weights.h5"

In [ ]:
data = prepare_data(TRAINING_DATA_PATH / "wide_win_idx")

In [ ]:
X_scaled = data["X_scaled"]
y_extended = data["y_extended"]
window_anchors = data["window_anchors"]
cell_ids_arr = data["cell_ids"]
scaler = data["scaler"]
cell_encoder = data["cell_encoder"]
kpi_columns = data["kpi_columns"]
seq_len = data["seq_len"]
feat_dim = data["feat_dim"]
n_classes = data["n_classes"]
output_dim = data["output_dim"]

In [ ]:
X_scaled.shape

In [ ]:
y_extended.shape

In [ ]:
window_anchors.shape

In [ ]:
cell_ids_arr.shape

In [ ]:
cell_ids_arr

In [ ]:
n_classes

In [ ]:
CVAE_LSTM_CONFIG = dict(
    latent_dim=32,
    hidden_dim=256,
    n_layers=2,
    use_attention=True,
    n_heads=4,
    beta=0.0,
    learning_rate=5e-4,
    epochs = 100,
    batch_size = 64,
    target_beta = 1.0,
    anneal_epochs = 20,
    lr_patience = 15,
    early_stop_patience = 30,
    min_lr = 1e-5
)

In [ ]:
arch, model = build_cvae_lstm_model(seq_len= seq_len,
                                    feat_dim= feat_dim,
                                    output_dim= output_dim,
                                    latent_dim = CVAE_LSTM_CONFIG["latent_dim"],
                                    hidden_dim = CVAE_LSTM_CONFIG["hidden_dim"],
                                    n_layers = CVAE_LSTM_CONFIG["n_layers"],
                                    use_attention = CVAE_LSTM_CONFIG["use_attention"],
                                    n_heads = CVAE_LSTM_CONFIG["n_heads"],
                                    beta = CVAE_LSTM_CONFIG["beta"],
                                    learning_rate = CVAE_LSTM_CONFIG["learning_rate"]
                                    )

In [ ]:
history = train_model(model = model,
                      X_scaled = X_scaled,
                      y_extended = y_extended,
                      weights_path = MODEL_PATH,
                      epochs = CVAE_LSTM_CONFIG["epochs"],
                      batch_size = CVAE_LSTM_CONFIG["batch_size"],
                      target_beta = CVAE_LSTM_CONFIG["target_beta"],
                      anneal_epochs = CVAE_LSTM_CONFIG["anneal_epochs"],
                      lr_patience = CVAE_LSTM_CONFIG["lr_patience"],
                      early_stop_patience = CVAE_LSTM_CONFIG["early_stop_patience"],
                      min_lr = CVAE_LSTM_CONFIG["min_lr"]
                      )

In [ ]:
# 4. Save
save_artifacts(RUN_DIR_PATH, model, data)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

In [ ]:
model, data = load_artifacts(RUN_DIR_PATH, MODEL_PATH)

In [ ]:
df = generate_timespan(model=model, **data,
    cell_id="bts_9/cell_9", date_start="2023-12-12", date_end="2024-03-12", seed=42)